# 10 — Multi-Horizon Training (6m, 1y)

Train 6-month and 1-year XGBoost models using the **exact same** blended ordinal loss pipeline as 3m (notebook 09).

Same: loss function, alpha, class weights, features, XGBoost params, boost rounds.
Different: label thresholds and horizon days (per CLAUDE.md).

**Output:** `src/ml/app/models/xgb_6m_blended.json` and `src/ml/app/models/xgb_1y_blended.json`

In [12]:
import json
import os
import re
import time
import warnings
from collections import Counter
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xgboost as xgb
import yfinance as yf
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    mean_absolute_error,
)

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
MODELS_DIR = Path("../src/ml/app/models")

OHLCV_CACHE = DATA_DIR / "sp500_ohlcv_5y.parquet"
GDELT_CACHE = DATA_DIR / "gdelt_sentiment_5y.parquet"

# Thresholds from CLAUDE.md (teacher-approved, adjusted for bull market)
# Split dates per horizon: chosen so each has a meaningful test window
# given that labeling requires `days` of future data beyond each sample.
THRESHOLDS = {
    "6m": {"days": 126, "bins": [-np.inf, -12, -4, 5, 18, np.inf], "split_date": "2025-04-01"},
    "1y": {"days": 252, "bins": [-np.inf, -15, -3, 10, 30, np.inf], "split_date": "2024-04-01"},
}
SIGNAL_LABELS = ["Strong Sell", "Sell", "Hold", "Buy", "Strong Buy"]
ALPHA = 0.3  # blended ordinal loss weight (same as 3m)

In [2]:
# ── Load or download OHLCV data ──
if OHLCV_CACHE.exists():
    print(f"Loading OHLCV cache from {OHLCV_CACHE}...")
    df_ohlcv = pd.read_parquet(OHLCV_CACHE)
    print(f"OHLCV: {len(df_ohlcv):,} rows, {df_ohlcv['Ticker'].nunique()} tickers")
else:
    print("OHLCV cache not found — downloading from yfinance...")

    # Fetch S&P 500 ticker list from Wikipedia
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "StockPredictor/1.0 (bachelor thesis project)"}
    response = requests.get(url, headers=headers)
    sp500_table = pd.read_html(StringIO(response.text))[0]
    sp500_tickers = sp500_table["Symbol"].str.replace(".", "-", regex=False).tolist()
    print(f"S&P 500 tickers: {len(sp500_tickers)}")

    # Batch download (same approach as notebook 05)
    BATCH_SIZE = 50
    MIN_ROWS = 504
    batches = [sp500_tickers[i:i + BATCH_SIZE] for i in range(0, len(sp500_tickers), BATCH_SIZE)]
    all_data = {}
    failed_tickers = []

    for batch_num, batch in enumerate(batches, 1):
        print(f"  Batch {batch_num}/{len(batches)}: {len(batch)} tickers...")
        try:
            raw = yf.download(batch, period="5y", group_by="ticker", auto_adjust=True)
            for ticker in batch:
                try:
                    df = raw[ticker].copy()
                    df = df.dropna(how="all")
                    if len(df) < MIN_ROWS:
                        failed_tickers.append((ticker, f"insufficient data: {len(df)} rows"))
                        continue
                    all_data[ticker] = df
                except KeyError:
                    failed_tickers.append((ticker, "not found"))
                except Exception as e:
                    failed_tickers.append((ticker, str(e)[:80]))
        except Exception as e:
            print(f"    Batch failed: {e}")
            for t in batch:
                failed_tickers.append((t, f"batch error"))
        if batch_num < len(batches):
            time.sleep(2)

    print(f"\nDownloaded {len(all_data)} tickers, {len(failed_tickers)} failed")

    # Convert to single DataFrame with Ticker column
    frames = []
    for ticker, df in all_data.items():
        df = df.copy()
        df["Ticker"] = ticker
        frames.append(df)
    df_ohlcv = pd.concat(frames)
    df_ohlcv.index.name = "Date"
    df_ohlcv.to_parquet(OHLCV_CACHE)
    print(f"Saved OHLCV cache to {OHLCV_CACHE}")
    print(f"OHLCV: {len(df_ohlcv):,} rows, {df_ohlcv['Ticker'].nunique()} tickers")

OHLCV cache not found — downloading from yfinance...
S&P 500 tickers: 503
  Batch 1/11: 50 tickers...


[*********************100%***********************]  50 of 50 completed


  Batch 2/11: 50 tickers...


[*********************100%***********************]  50 of 50 completed


  Batch 3/11: 50 tickers...


...

  Batch 10/11: 50 tickers...


[*********************100%***********************]  50 of 50 completed


  Batch 11/11: 3 tickers...


[*********************100%***********************]  3 of 3 completed



Downloaded 501 tickers, 2 failed
Saved OHLCV cache to data\sp500_ohlcv_5y.parquet
OHLCV: 625,945 rows, 501 tickers


In [9]:
# ── Load sentiment data and map orgs to tickers ──
if GDELT_CACHE.exists():
    gdelt_raw = pd.read_parquet(GDELT_CACHE)
    print(f"GDELT raw: {len(gdelt_raw):,} rows")

    # Normalize: article_date -> Date, tone -> clipped sentiment
    gdelt_raw["article_date"] = pd.to_datetime(gdelt_raw["article_date"], format="%Y%m%d")
    gdelt_raw["sentiment"] = np.clip(gdelt_raw["tone"] / 10, -1, 1)

    # Wikipedia S&P 500 table for org -> ticker mapping
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "StockPredictor/1.0 (bachelor thesis project)"}
    resp = requests.get(url, headers=headers)
    sp500_table = pd.read_html(StringIO(resp.text))[0]

    def clean_company_name(name):
        name = name.lower().strip().replace(",", " ").replace(".", " ")
        name = re.sub(r"\s+", " ", name).strip()
        name = re.sub(r"^the ", "", name)
        name = re.sub(r"\s*&\s*co\s*$", "", name)
        name = re.sub(
            r"\s+(inc|corp|co|ltd|llc|plc|lp|nv|sa|corporation|incorporated|"
            r"company|companies|holdings|group|enterprises?|international)\s*$",
            "", name,
        )
        return name.strip()

    # Build org -> ticker mapping
    ohlcv_tickers = set(df_ohlcv["Ticker"].unique())
    company_to_tickers = {}
    for _, row in sp500_table.iterrows():
        ticker = row["Symbol"].replace(".", "-")
        if ticker not in ohlcv_tickers:
            continue
        cleaned = clean_company_name(row["Security"])
        company_to_tickers.setdefault(cleaned, []).append(ticker)

    mapping_rows = []
    for company, ticker_list in company_to_tickers.items():
        for ticker in ticker_list:
            mapping_rows.append({"matched_org": company, "ticker": ticker})
    mapping_df = pd.DataFrame(mapping_rows)

    # Join GDELT with ticker mapping
    scored_df = gdelt_raw.merge(mapping_df, on="matched_org", how="inner")
    scored_df = scored_df[["ticker", "article_date", "sentiment"]].rename(
        columns={"article_date": "date"}
    )
    print(f"Matched {scored_df['ticker'].nunique()} tickers in GDELT")
    print(f"Scored rows: {len(scored_df):,}")

    # Build per-ticker daily sentiment lookup
    sentiment_by_ticker = {}
    for ticker in scored_df["ticker"].unique():
        ticker_news = scored_df[scored_df["ticker"] == ticker]
        daily = ticker_news.groupby("date").agg(
            daily_sentiment=("sentiment", "mean"),
            daily_count=("sentiment", "count"),
        )
        sentiment_by_ticker[ticker] = daily
    print(f"Sentiment lookup built for {len(sentiment_by_ticker)} tickers")
else:
    sentiment_by_ticker = {}
    print("No sentiment cache — will train without sentiment features")

GDELT raw: 66,466,075 rows
Matched 426 tickers in GDELT
Scored rows: 66,466,075
Sentiment lookup built for 426 tickers


In [10]:
# ── Feature engineering ──
import sys
sys.path.insert(0, str(Path("../src/ml")))
from app.services.feature_engineering import compute_features

def build_features(ticker: str, ticker_df: pd.DataFrame) -> pd.DataFrame:
    """Compute 19 technical + 3 sentiment features for one ticker."""
    df = compute_features(ticker_df.copy())

    if ticker in sentiment_by_ticker:
        daily = sentiment_by_ticker[ticker]
        daily_aligned = daily.reindex(df.index)
        daily_aligned["daily_sentiment"] = daily_aligned["daily_sentiment"].ffill()
        daily_aligned["daily_count"] = daily_aligned["daily_count"].fillna(0)

        sent_avg_20 = daily_aligned["daily_sentiment"].rolling(20, min_periods=5).mean()
        sent_avg_60 = daily_aligned["daily_sentiment"].rolling(60, min_periods=10).mean()
        sent_vol_20 = daily_aligned["daily_count"].rolling(20, min_periods=1).sum()

        df["sentiment_avg_20d"] = sent_avg_20
        df["sentiment_volume_20d"] = sent_vol_20
        df["sentiment_momentum"] = sent_avg_20 - sent_avg_60

    return df

# Build features for all tickers
all_features = []
tickers = df_ohlcv["Ticker"].unique()
for i, ticker in enumerate(tickers):
    if (i + 1) % 100 == 0:
        print(f"  {i + 1}/{len(tickers)}")
    tdf = df_ohlcv[df_ohlcv["Ticker"] == ticker].copy()
    tdf = tdf.sort_index()
    featured = build_features(ticker, tdf)
    featured["Ticker"] = ticker
    all_features.append(featured)

df_all = pd.concat(all_features)
coverage = df_all["sentiment_avg_20d"].notna().mean() * 100
print(f"Feature matrix: {len(df_all):,} rows")
print(f"Sentiment coverage: {coverage:.1f}%")

  100/501
  200/501
  300/501
  400/501
  500/501
Feature matrix: 625,945 rows
Sentiment coverage: 83.2%


In [13]:
FEATURE_COLS = [
    "Price_SMA50_Ratio", "Price_SMA200_Ratio", "Price_EMA20_Ratio",
    "MACD_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm",
    "RSI", "Stoch_K", "Stoch_D", "ROC",
    "BB_Position", "BB_Width", "ATR_Pct",
    "OBV_Norm", "Vol_SMA_Ratio",
    "Dist_52w_High", "Dist_52w_Low", "Return_1m", "Return_3m",
    "sentiment_avg_20d", "sentiment_volume_20d", "sentiment_momentum",
]

n_classes = len(SIGNAL_LABELS)
le = LabelEncoder()
le.fit(SIGNAL_LABELS)


# ── Blended ordinal loss (EXACT copy from notebook 09) ──
def blended_ordinal_obj(alpha=0.3):
    """Returns a custom XGBoost objective that blends softmax + ordinal loss.

    alpha=0 -> pure softmax (standard multi:softprob behavior)
    alpha=1 -> pure ordinal
    alpha=0.3 -> 70% softmax + 30% ordinal penalty
    """
    def objective(preds, dtrain):
        labels = dtrain.get_label().astype(int)
        n_samples = len(labels)
        n_cls = 5
        preds = preds.reshape(n_samples, n_cls)

        # Softmax probabilities
        preds_shifted = preds - preds.max(axis=1, keepdims=True)
        exp_p = np.exp(preds_shifted)
        probs = exp_p / exp_p.sum(axis=1, keepdims=True)

        # Part 1: Standard softmax gradient
        one_hot = np.zeros_like(probs)
        one_hot[np.arange(n_samples), labels] = 1.0
        grad_softmax = probs - one_hot
        hess_softmax = probs * (1.0 - probs)

        # Part 2: Ordinal distance penalty (absolute, not squared)
        classes = np.arange(n_cls)
        dist_abs = np.abs(classes[np.newaxis, :] - labels[:, np.newaxis]).astype(float)
        expected_dist = (dist_abs * probs).sum(axis=1, keepdims=True)
        grad_ordinal = probs * (dist_abs - expected_dist)
        hess_ordinal = probs * (1.0 - probs) + 1e-6

        # Blend
        grad = (1 - alpha) * grad_softmax + alpha * grad_ordinal
        hess = (1 - alpha) * hess_softmax + alpha * hess_ordinal
        hess = np.maximum(hess, 1e-6)

        return grad.flatten(), hess.flatten()

    return objective


# ── XGBoost params (same as notebook 09) ──
native_params = {
    "max_depth": 6,
    "learning_rate": 0.1,
    "num_class": n_classes,
    "tree_method": "hist",
    "random_state": 42,
    "nthread": -1,
}

results = {}

for horizon, cfg in THRESHOLDS.items():
    split_date = cfg["split_date"]
    print(f"\n{'='*60}")
    print(f"TRAINING: {horizon} horizon ({cfg['days']} trading days)")
    print(f"Split date: {split_date}")
    print(f"{'='*60}")

    # Label: future return over horizon window
    df_h = df_all.copy()
    df_h["future_return"] = df_h.groupby("Ticker")["Close"].transform(
        lambda s: s.shift(-cfg["days"]) / s - 1
    ) * 100
    df_h = df_h.dropna(subset=["future_return"])
    df_h["signal"] = pd.cut(
        df_h["future_return"],
        bins=cfg["bins"],
        labels=SIGNAL_LABELS,
    )
    df_h = df_h.dropna(subset=["signal"])
    df_h["label"] = le.transform(df_h["signal"])

    # Class distribution
    print("\nClass distribution:")
    dist = df_h["signal"].value_counts(normalize=True).sort_index()
    for sig, pct in dist.items():
        print(f"  {sig}: {pct:.1%}")

    # Time-based split
    train = df_h[df_h.index < split_date]
    test = df_h[df_h.index >= split_date]
    print(f"\nTrain: {len(train):,} | Test: {len(test):,}")

    X_train = train[FEATURE_COLS].values
    y_train = train["label"].values
    X_test = test[FEATURE_COLS].values
    y_test = test["label"].values

    # Class weights (same as notebook 09)
    class_counts = Counter(y_train)
    total = len(y_train)
    class_weights_dict = {cls: total / (n_classes * count) for cls, count in class_counts.items()}
    sample_weights = np.array([class_weights_dict[y] for y in y_train])

    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights, feature_names=FEATURE_COLS)
    dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURE_COLS)

    obj = blended_ordinal_obj(alpha=ALPHA)
    booster = xgb.train(
        native_params,
        dtrain,
        num_boost_round=200,
        obj=obj,
    )

    # Evaluate
    raw = booster.predict(dtest, output_margin=True).reshape(-1, n_classes)
    shifted = raw - raw.max(axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    probs = exp_vals / exp_vals.sum(axis=1, keepdims=True)
    y_pred = probs.argmax(axis=1)

    f1 = f1_score(y_test, y_pred, average="weighted")
    acc = accuracy_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print(f"\nResults:")
    print(f"  Weighted F1: {f1:.4f}")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  MAE:         {mae:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=SIGNAL_LABELS))
    print(f"Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Distance breakdown (same as notebook 09)
    distances = np.abs(y_test - y_pred)
    print("\nPrediction distance breakdown:")
    for d in range(5):
        pct = (distances == d).mean() * 100
        print(f"  Distance {d}: {pct:.1f}%")
    print(f"  Within 1 class: {(distances <= 1).mean()*100:.1f}%")

    # Save model
    model_path = MODELS_DIR / f"xgb_{horizon}_blended.json"
    booster.save_model(str(model_path))
    print(f"\nModel saved to {model_path}")

    results[horizon] = {
        "weighted_f1": round(f1, 4),
        "accuracy": round(acc, 4),
        "mae": round(mae, 4),
        "train_samples": len(train),
        "test_samples": len(test),
        "split_date": split_date,
    }

print("\n\nSummary:")
for h, r in results.items():
    print(f"  {h}: F1={r['weighted_f1']:.4f}  Acc={r['accuracy']:.4f}  MAE={r['mae']:.4f}")


TRAINING: 6m horizon (126 trading days)
Split date: 2025-04-01

Class distribution:
  Strong Sell: 18.1%
  Sell: 13.8%
  Hold: 19.3%
  Buy: 24.7%
  Strong Buy: 24.1%

Train: 495,185 | Test: 67,634

Results:
  Weighted F1: 0.2744
  Accuracy:    0.3058
  MAE:         1.3919

Classification Report:
              precision    recall  f1-score   support

 Strong Sell       0.27      0.36      0.31     15663
        Sell       0.22      0.26      0.24     11883
        Hold       0.14      0.04      0.06      8493
         Buy       0.40      0.54      0.46     20213
  Strong Buy       0.20      0.06      0.09     11382

    accuracy                           0.31     67634
   macro avg       0.25      0.25      0.23     67634
weighted avg       0.27      0.31      0.27     67634

Confusion Matrix:
[[ 5638  3770   568  5124   563]
 [ 4203  3146   329  3911   294]
 [ 3090  2316   305  2504   278]
 [ 4274  2897   641 10933  1468]
 [ 3459  2330   333  4600   660]]

Prediction distance breakdow

In [14]:
metadata_path = MODELS_DIR / "training_metadata.json"
with open(metadata_path) as f:
    metadata = json.load(f)

for horizon, res in results.items():
    key = f"sentiment_model_{horizon}"
    metadata[key] = {
        "horizon": horizon,
        "feature_columns": FEATURE_COLS,
        "n_features": len(FEATURE_COLS),
        "sentiment_features": ["sentiment_avg_20d", "sentiment_volume_20d", "sentiment_momentum"],
        "split_date": res["split_date"],
        "results": {k: v for k, v in res.items() if k != "split_date"},
    }

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Updated training_metadata.json")

Updated training_metadata.json
